In [ ]:
#Q.10 행동 로그와 주문 데이터 대사
# 1. orders를 web 채널로 한정해 web_logs의 purchase 이벤트 수/고객수와 유효 주문수/고객수를 기간별로 대조
#    (로그는 사이트 행동 기록이므로 store/app 주문과 대조하는 것은 무의미 -> 근거를 수치로 보여야 함)
# 2. 고객x날짜 단위로 두 소스를 병합해 '로그에만 있는 구매'와 '주문에만 있는 구매' 규모 추정
# 3. 로그 유실(또는 과다) 추정치 제시 + 로그 기반 지표 사용 가이드라인 3줄


#포인트 1: 로그는 사이트 한정기록이므로 orders를 web채널로 한정해야됨 
#포인트 2: 고객 X 날짜 병합해서 로그에만 있는 구매 vs 주문에만 있는 구매를 대조

In [ ]:
import pandas as pd

orders_path = "../data/orders.csv"
logs_path = "../data/web_logs.csv"

orders = pd.read_csv(
    orders_path,
    dtype={"order_id": "int64", "customer_id": "int64", "channel": "category", "status": "category"},
)
# order_datetime 타입 오염(문자열 사이에 결측 float(NaN)이 섞여 object dtype) -> errors='coerce'로 강제 파싱, 실패/결측은 NaT
orders["order_datetime"] = pd.to_datetime(orders["order_datetime"], errors="coerce")
print("orders 결측 order_datetime:", orders["order_datetime"].isna().sum(), "/", len(orders))

logs = pd.read_csv(
    logs_path,
    usecols=["customer_id", "event_time", "event_type"],
    dtype={"event_type": "category"},
)
logs["event_time"] = pd.to_datetime(logs["event_time"])
purchase = logs[logs["event_type"] == "purchase"].copy()
print("purchase 이벤트 수:", len(purchase))
print(orders.head())
purchase.head()

### 왜 web 채널로 한정하는가

`web_logs`는 사이트(웹)에서 발생한 행동만 기록한다. store/app 채널 주문은 애초에 이 로그 시스템을 거치지 않으므로, 전체 채널 주문과 비교하면 store/app 주문이 통째로 "로그 유실"처럼 잡혀 유실률이 실제보다 크게 부풀려진다. 아래 셀에서 전체 채널 대비 순수 총량 비교가 얼마나 왜곡되는지 직접 확인한다.

In [ ]:
print("채널별 주문 수:")
print(orders["channel"].value_counts())
print()

naive_loss = 1 - len(purchase) / len(orders)
print(f"[잘못된 비교] 전체 채널 주문 대비 purchase 로그 커버리지: {1 - naive_loss:.1%}  (유실률 {naive_loss:.1%})")
print("-> store/app 주문(전체의 45%)이 애초에 로그 대상이 아니므로 이 숫자는 의미가 없다.")
print()

web_orders = orders[orders["channel"] == "web"]
print(f"web 채널 주문 수: {len(web_orders):,} (전체의 {len(web_orders)/len(orders):.1%})")

### 대사 스코프 정의 규칙 (핵심 판단)

1. **채널**: `channel == "web"`만 사용한다(위에서 근거 확인).
2. **유효 주문 = status != "canceled"**: 취소된 주문은 결제/배송으로 이어지지 않아 매출이 확정되지 않으므로 재무 관점의 "유효 주문"에서 제외한다. `returned`(반품)는 일단 결제·매출이 발생했던 건이므로 포함한다.
3. **기간 스코프를 로그 존재 기간으로 맞춘다**: `web_logs`는 2024-01 ~ 2024-06까지만 존재하는데 `orders`는 2025-06까지 이어진다. 로그가 아예 없는 기간(2024-07 이후)의 주문을 분모에 넣으면 "유실"이 아니라 "로그 자체가 없는 기간"이 섞여 착시를 일으키므로, 두 소스가 겹치는 기간만 비교한다.
4. **customer_id 결측 로그는 대사 불가**: purchase 로그 중 상당수가 customer_id 결측이라 어떤 주문과도 매칭할 수 없다. 이 구간은 매칭 시도에서 제외하고 별도 건수로만 보고한다(로그 신뢰도를 낮추는 요인으로 가이드라인에 반영).
5. **매칭 단위 = 고객x일(day) 단위 '도달 여부'**: 로그의 purchase 건수와 주문 건수가 세션 분할/재시도 등으로 1:1이 아닐 수 있으므로, 개별 건 단위로 짝짓지 않고 "같은 고객이 같은 날짜에 로그/주문이 존재하는가"로만 매칭한다.

In [ ]:
# 스코프 적용: web + 유효주문(취소 제외)
valid_orders = orders[(orders["channel"] == "web") & (orders["status"] != "canceled")].copy()

# 기간 스코프: 로그가 실제로 존재하는 기간으로 제한
log_start = purchase["event_time"].min().normalize()
log_end = purchase["event_time"].max().normalize() + pd.Timedelta(days=1)
print(f"로그 존재 기간: {log_start.date()} ~ {log_end.date()}")
print(f"주문 존재 기간(all): {orders['order_datetime'].min()} ~ {orders['order_datetime'].max()}")

before_scope = len(valid_orders)
scoped_orders = valid_orders[(valid_orders["order_datetime"] >= log_start) & (valid_orders["order_datetime"] < log_end)].copy()
print(f"유효 web 주문 {before_scope:,}건 중 로그 기간 밖 주문 {before_scope - len(scoped_orders):,}건 제외 -> 스코프 내 {len(scoped_orders):,}건")
print()

# customer_id 결측 purchase 로그는 매칭 불가 -> 별도 계상
n_missing_cust = purchase["customer_id"].isna().sum()
purchase_matchable = purchase.dropna(subset=["customer_id"]).copy()
purchase_matchable["customer_id"] = purchase_matchable["customer_id"].astype(int)
print(f"purchase 로그 {len(purchase):,}건 중 customer_id 결측 {n_missing_cust:,}건 ({n_missing_cust/len(purchase):.1%}) -> 대사 불가로 별도 계상")
print(f"매칭 가능 purchase 로그: {len(purchase_matchable):,}건")

In [ ]:
# 1) 총량 대조
print("=== 총량 대조 (web / 유효주문 / 로그기간 내) ===")
print(f"purchase 로그 건수(매칭가능): {len(purchase_matchable):,}  |  로그 고객수: {purchase_matchable['customer_id'].nunique():,}")
print(f"유효 web 주문 건수: {len(scoped_orders):,}  |  주문 고객수: {scoped_orders['customer_id'].nunique():,}")
coverage = len(purchase_matchable) / len(scoped_orders)
print(f"총량 커버리지(로그/주문): {coverage:.1%}  (주문 대비 부족분 {len(scoped_orders)-len(purchase_matchable):,}건)")

In [ ]:
# 2) 월별 대조
purchase_matchable["month"] = purchase_matchable["event_time"].dt.to_period("M")
scoped_orders["month"] = scoped_orders["order_datetime"].dt.to_period("M")

monthly_log = purchase_matchable.groupby("month").agg(log_purchase=("customer_id", "count"), log_customers=("customer_id", "nunique"))
monthly_order = scoped_orders.groupby("month").agg(order_count=("customer_id", "count"), order_customers=("customer_id", "nunique"))

monthly = monthly_log.join(monthly_order, how="outer")
monthly["gap"] = monthly["order_count"] - monthly["log_purchase"]
monthly["log_coverage"] = (monthly["log_purchase"] / monthly["order_count"]).round(3)
monthly

월별 커버리지가 1월 76.8% -> 6월 40.6%로 꾸준히 떨어진다. 로그 건수는 매달 8,500~9,000건대로 거의 고정인데, 유효 주문은 11,600건 -> 21,200건으로 계속 늘고 있다. 즉 로그 유실은 특정 달의 일시적 사고가 아니라 **시간이 갈수록 커지는 구조적 문제**로 보인다.

In [ ]:
# 3) 고객x일 단위 병합 (merge indicator) - 매칭 단위 = 같은 고객, 같은 날짜에 둘 다 존재하는가
purchase_matchable["date"] = purchase_matchable["event_time"].dt.normalize()
scoped_orders["date"] = scoped_orders["order_datetime"].dt.normalize()

log_cd = purchase_matchable.groupby(["customer_id", "date"]).size().reset_index(name="log_cnt")
order_cd = scoped_orders.groupby(["customer_id", "date"]).size().reset_index(name="order_cnt")

merged = log_cd.merge(order_cd, on=["customer_id", "date"], how="outer", indicator=True)
merge_counts = merged["_merge"].value_counts()
print(merge_counts)
print()

n_both = merge_counts["both"]
n_order_only = merge_counts["right_only"]
n_log_only = merge_counts["left_only"]

print(f"주문에만 있는 고객x일: {n_order_only:,}건 ({n_order_only/(n_both+n_order_only):.1%} of 주문 고객x일)")
print(f"로그에만 있는 고객x일: {n_log_only:,}건 ({n_log_only/(n_both+n_log_only):.1%} of 로그 고객x일)")
print(f"양쪽 다 있는 고객x일: {n_both:,}건")

In [ ]:
# 참고: orders customer_id 중 로그 고객 ID 범위(1000~5999대)를 벗어난 비정상 값 규모
anomalous_cust = (scoped_orders["customer_id"] > 10000).sum()
print(f"스코프 내 주문 중 비정상 customer_id(>10000) 건수: {anomalous_cust:,} ({anomalous_cust/len(scoped_orders):.2%})")
print("-> 이 건들은 애초에 로그 고객 ID 공간과 겹치지 않아 구조적으로 매칭이 불가능하다(별도 데이터 품질 이슈로 보고).")

### 로그 유실 추정치

- **총량 기준**: web·유효주문·로그기간 스코프에서 로그는 주문의 약 **53%**만 커버한다(부족분 약 45,500건).
- **월별 기준**: 이 커버리지는 고정값이 아니라 1월 76.8% -> 6월 40.6%로 계속 악화되는 추세다. 주문은 늘어나는데 로그는 늘지 않아 유실이 누적되고 있다.
- **고객x일 기준(가장 보수적, 가장 신뢰도 높은 추정)**: 실제로 같은 고객·같은 날짜에 로그와 주문이 동시에 존재하는 비율은 주문 고객x일 기준 **4.6%**, 로그 고객x일 기준 **7.9%**에 불과하다. 총량 비율(53%)만 보면 "로그가 주문의 절반 정도는 담아낸다"고 오해하기 쉽지만, 실제 개별 고객·일 단위로 들어가면 서로 거의 겹치지 않는다 — 총량이 비슷한 규모로 나온 것은 두 지표가 같은 사건을 추적해서가 아니라 우연히 규모가 비슷했을 뿐일 가능성이 크다.
- 추가로 purchase 로그의 34.9%는 customer_id 결측으로 애초에 대사가 불가능했고, 주문 쪽에도 로그 고객 ID 범위를 벗어난 비정상 customer_id가 0.47% 존재해 완전한 매칭은 원천적으로 불가능하다.

### 로그 기반 지표 사용 가이드라인 (3줄)

1. 로그 기반 "구매 이벤트 수/전환율"은 **월별 방향성(추세) 참고용**으로만 쓰고, 매출·주문수 확정치는 항상 주문 시스템 값을 기준으로 한다 — 커버리지가 40~77% 사이에서 계속 변하므로 절대값으로 신뢰할 수 없다.
2. 로그와 주문을 비교할 때는 반드시 **channel=web, status≠canceled, 로그 존재 기간**으로 스코프를 맞춘 뒤에만 비교한다 — 스코프를 안 맞추면 유실률이 실제보다 더 부풀려지거나 왜곡된다.
3. 개별 고객·개별 거래 단위의 "로그=주문" 매칭은 성립하지 않는다(고객x일 매칭률 4.6~7.9%) — 로그는 개별 거래 검증이 아니라 "행동이 있었다"는 근사 신호로만 사용하고, customer_id 결측(34.9%) 구간은 대사 자체가 불가능하다는 점을 항상 각주로 명시한다.